# Tutorial: Plan 004.1 Query Foundation

**Audience:** readers with basic Python knowledge who want to inspect the first real query-grammar infrastructure.

**Learning goals:**

- distinguish the unchanged public API from the new private implementation objects;
- inspect an immutable `CurrentObservationQuery`;
- compile that description into parameterized SQLite;
- execute it and materialize `FileObservation` objects;
- see that current state is chosen at execution time.

**Outline:** setup → public behavior → description → compilation → execution → current-state exercise.

Names beginning with `_` are private. This notebook uses them for learning, not as supported application APIs. Plan 004.1 intentionally adds no public grammar.


In [ ]:
from __future__ import annotations

from dataclasses import FrozenInstanceError, fields, is_dataclass
from pathlib import Path
from tempfile import TemporaryDirectory
from textwrap import dedent

import archiver
from archiver import Catalog, Location
from archiver._query import CurrentObservationQuery
from archiver._sqlite_query import compile_current_observations, execute_current_observations


temporary_directory = TemporaryDirectory(prefix="archiver-query-foundation-")
workspace = Path(temporary_directory.name)
root = workspace / "source"
(root / "nested").mkdir(parents=True)

source_paths = (
    root / "empty.bin",
    root / "nested" / "a-copy.bin",
    root / "z-copy.bin",
)
source_paths[0].touch()
source_paths[1].write_bytes(b"same content")
source_paths[2].write_bytes(b"same content")

catalog = Catalog.create(workspace / "catalog.sqlite")
scan_summary = catalog.scan_directory(root)
scan_summary


## 1. The public API did not change

`Catalog.current_files(root)` still returns a list of `FileObservation` objects for the latest successful scan, ordered by relative path. Callers do not need to know about the new internal layers.


In [ ]:
public_rows = catalog.current_files(root)

[
    (row.relative_path.as_posix(), row.size_bytes, row.content_id.digest[:12])
    for row in public_rows
]


## 2. The query description contains meaning, not SQL

`CurrentObservationQuery` is a frozen dataclass. Its single field is the persisted `Location` to query. Ordering by relative path is part of this query type's meaning.

It contains no SQL statement, parameters, cursor, or database connection. That separation lets the description stay small and backend-independent.


In [ ]:
location = public_rows[0].location
query = CurrentObservationQuery(location=location)

try:
    setattr(query, "location", Location(id=999, root=root))
except FrozenInstanceError as error:
    mutation_result = type(error).__name__

{
    "query": query,
    "is a dataclass": is_dataclass(query),
    "dataclass fields": [field.name for field in fields(query)],
    "mutation attempt": mutation_result,
    "contains SQL": hasattr(query, "statement"),
    "publicly exported": hasattr(archiver, "CurrentObservationQuery"),
}


In [ ]:
?CurrentObservationQuery

## 3. Compilation translates meaning into SQLite

The SQLite compiler accepts the semantic description and returns another immutable object: `SqliteQuery`.

Its statement uses `?` as a placeholder. The location ID travels separately in `parameters`; it is not pasted into the SQL text.


In [ ]:
compiled = compile_current_observations(query)

print(dedent(compiled.statement).strip())
print("parameters:", compiled.parameters)

assert "locations.id = ?" in compiled.statement
assert compiled.parameters == (location.id,)
assert str(root) not in compiled.statement


## 4. Execution materializes domain objects

The executor compiles the description, runs the statement on the existing catalog connection, and converts each SQLite row directly into a `FileObservation`.

Using `catalog._connection` is appropriate only for this internal walkthrough. Normal callers continue to use `catalog.current_files(root)`.


In [ ]:
def source_snapshot(paths: tuple[Path, ...]) -> dict[Path, tuple[bytes, int]]:
    return {path: (path.read_bytes(), path.stat().st_mtime_ns) for path in paths}


before_query = source_snapshot(source_paths)
internal_rows = execute_current_observations(catalog._connection, query)
after_query = source_snapshot(source_paths)

assert internal_rows == public_rows
assert after_query == before_query
assert all(type(row).__name__ == "FileObservation" for row in internal_rows)

for fo in internal_rows:
    #print (fo.relative_path)
    #print(fo.content_id)
    print(fo.relative_path.as_posix(), fo.size_bytes, fo.content_id.digest[:12])
    #print('\n')


## 5. The same description reads current state later

The query stores a location, not a snapshot or scan ID. We can create it once, complete another successful scan, and execute it again. SQLite follows `locations.current_scan_id`, so older observations remain historical while results show the new current state.


In [ ]:
(root / "z-copy.bin").rename(root / "b-renamed.bin")
catalog.scan_directory(root)

second_internal_rows = execute_current_observations(catalog._connection, query)
second_public_rows = catalog.current_files(root)
history = list(catalog.observation_history())

assert second_internal_rows == second_public_rows
assert query.location == location
assert len(history) == 6

{
    "same query object": query,
    "first current paths": [row.relative_path.as_posix() for row in public_rows],
    "second current paths": [row.relative_path.as_posix() for row in second_internal_rows],
    "historical observation count": len(history),
}


## Exercise: protect the compiler boundary

`CurrentObservationQuery` requires a persisted location ID. Predict what happens if you try to construct one with `Location(id=None, ...)`.

Then explain why validation belongs before compilation: should invalid domain input become SQL parameters at all?


In [ ]:
# Solution — try this yourself before running the cell.
try:
    CurrentObservationQuery(location=Location(id=None, root=root))
except ValueError as error:
    validation_message = str(error)

assert validation_message == "a current-observation query requires a persisted location"
validation_message


In [ ]:
# Cleanup: the tutorial leaves no catalog or source files behind.
catalog.close()
temporary_directory.cleanup()


## The Plan 004.1 boundary

```text
Catalog.current_files(root)
        ↓
CurrentObservationQuery(location)     # semantic, immutable description
        ↓
compile_current_observations(...)    # parameterized SQLite statement
        ↓
execute_current_observations(...)    # FileObservation materialization
```

Plan 004.1 proves this narrow path while preserving behavior. It does **not** add fields, predicates, filtering, configurable sorting, limits, grouping, or a public fluent API.

**Common pitfalls:** private names may change; a query description is not a result; this query observes current state when executed rather than freezing a snapshot.

**Next experiment:** compare this real foundation with the separate toy grammar notebook, but do not assume the toy `Field` and `Predicate` names are future public APIs.
